<a href="https://colab.research.google.com/github/Balachandar-Ganesan/DeepLearning/blob/main/200_5_BuildYourOwnLLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [36]:
!wget https://raw.githubusercontent.com/Balachandar-Ganesan/DeepLearning/refs/heads/main/TinyStories-200.txt

--2026-03-06 11:28:23--  https://raw.githubusercontent.com/Balachandar-Ganesan/DeepLearning/refs/heads/main/TinyStories-200.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 23527 (23K) [text/plain]
Saving to: ‘TinyStories-200.txt.1’

TinyStories-200.txt 100%[===================>]  22.98K  --.-KB/s    in 0s      

2026-03-06 11:28:24 (78.8 MB/s) - ‘TinyStories-200.txt.1’ saved [23527/23527]



In [37]:
!pip3 install tiktoken
!pip3 list | grep tiktoken

tiktoken                                 0.12.0


In [38]:
book_contents = ""
with open("/content/TinyStories-200.txt", "r") as f:
  book_contents = f.read()

In [39]:
import tiktoken
encoding = tiktoken.get_encoding("o200k_base")

In [40]:
encoding.encode("Timing beats Speed. Precision beats Power")




[81398, 54439, 23451, 13, 87969, 54439, 10079]

In [41]:
encoding.decode([54439])


' beats'

In [42]:
def generate_training_data(data, n, tokenizer):
    tokens = tokenizer.encode(data,disallowed_special=())
    X = []
    y = []
    for i in range(len(tokens) - n):
      X.append(tokens[i : n + i])
      y.append(tokens[i + 1 : n + i + 1])

    return [X, y]

In [43]:
sequence_len = 5
_X, _y = generate_training_data(book_contents, sequence_len, encoding)

In [44]:
import torch

tensor_X = torch.tensor(_X, dtype = torch.long)
tensor_y = torch.tensor(_y, dtype = torch.long)

#tensor_X = tensor_X.to("cuda")
#tensor_y = tensor_y.to("cuda")


In [45]:
from torch.utils.data import TensorDataset, DataLoader

dataset = TensorDataset(tensor_X, tensor_y)
dataloader = DataLoader(dataset, batch_size=256, shuffle=True)

In [46]:
import torch.nn as nn

In [47]:
class TinyLLM(nn.Module):
  def __init__(self, vocab_size, embed_size, hidden_size):
    super(TinyLLM, self).__init__()
    self.embedding = nn.Embedding(vocab_size, embed_size)
    self.rnn = nn.RNN(embed_size, hidden_size, batch_first=True)
    self.fc = nn.Linear(hidden_size, vocab_size)

  def forward(self, x):
    out = self.embedding(x)
    out, _ = self.rnn(out)
    out = self.fc(out)
    return out


#TinyLLM.forward = forward

In [48]:
embed_size = 128
hidden_size = 256

model = TinyLLM(encoding.n_vocab,
                embed_size, hidden_size
)

In [49]:
num_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {num_params}")

Total parameters: 77106131


In [50]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)


In [51]:
n_epochs = 40

for epoch in range(n_epochs):
  epoch_loss = 0
  for X, y in dataloader:
    optimizer.zero_grad()
    outputs = model(X)
    outputs = outputs.view(-1, encoding.n_vocab)
    y = y.view(-1)
    loss = criterion(outputs, y)
    loss.backward()
    optimizer.step()
    epoch_loss += loss.item()

  avg_loss = epoch_loss / len(dataloader)
  print(f'Epoch [{epoch+1}/{n_epochs}], Loss: {avg_loss:.4f}')

Epoch [1/40], Loss: 10.3422
Epoch [2/40], Loss: 5.9006
Epoch [3/40], Loss: 4.7530
Epoch [4/40], Loss: 4.2640
Epoch [5/40], Loss: 3.8617
Epoch [6/40], Loss: 3.4904
Epoch [7/40], Loss: 3.1427
Epoch [8/40], Loss: 2.8107
Epoch [9/40], Loss: 2.4990
Epoch [10/40], Loss: 2.2137
Epoch [11/40], Loss: 1.9583
Epoch [12/40], Loss: 1.7391
Epoch [13/40], Loss: 1.5556
Epoch [14/40], Loss: 1.4004
Epoch [15/40], Loss: 1.2687
Epoch [16/40], Loss: 1.1578
Epoch [17/40], Loss: 1.0626
Epoch [18/40], Loss: 0.9813
Epoch [19/40], Loss: 0.9128
Epoch [20/40], Loss: 0.8562
Epoch [21/40], Loss: 0.8073
Epoch [22/40], Loss: 0.7671
Epoch [23/40], Loss: 0.7333
Epoch [24/40], Loss: 0.7049
Epoch [25/40], Loss: 0.6813
Epoch [26/40], Loss: 0.6618
Epoch [27/40], Loss: 0.6442
Epoch [28/40], Loss: 0.6295
Epoch [29/40], Loss: 0.6169
Epoch [30/40], Loss: 0.6074
Epoch [31/40], Loss: 0.5966
Epoch [32/40], Loss: 0.5884
Epoch [33/40], Loss: 0.5813
Epoch [34/40], Loss: 0.5755
Epoch [35/40], Loss: 0.5698
Epoch [36/40], Loss: 0.5645


In [52]:
torch.save(model.state_dict(), "tinyllm40_weights.pth")


In [53]:
model.eval()

TinyLLM(
  (embedding): Embedding(200019, 128)
  (rnn): RNN(128, 256, batch_first=True)
  (fc): Linear(in_features=256, out_features=200019, bias=True)
)

In [54]:
import torch.nn.functional as F

def generate_text(prompt, tokenizer, max_length = 50):
  prompt = tokenizer.encode(prompt)[-sequence_len:]
  generated = prompt.copy()
  with torch.no_grad():
    for _ in range(max_length):
      current = [generated[-sequence_len:]]
      current = torch.tensor(current, dtype=torch.long)
      output = model(current)
      predictions = output[:, -1, :]
      probabilities = F.softmax(predictions, dim=-1)
      next_token = torch.multinomial(probabilities, num_samples=1).item()
      generated.append(next_token)
  print(encoding.decode(generated))

In [58]:
generate_text("neighbour", encoding, max_length=25)


neighbour <|endoftext|>
A father and son's relationship is tested by their differing ideologies on tradition versus modernity <


In [59]:
generate_text("Mother", encoding, max_length=25)


Motherayas in work shared apartment find an abandoned baby and must learn how to take care of a real crime <|endoftext
